In [ ]:
import numpy as np
import pandas as pd
import scipy.sparse as sp
import matplotlib.pyplot as plt
import seaborn as sns
from sknetwork.data import load_netset
from sknetwork.embedding import SVD
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.metrics import roc_auc_score, roc_curve, precision_recall_fscore_support

RNG_SEED = 42
FEATURE_NAMES = ['out_deg_u', 'in_deg_v', 'in_deg_u', 'out_deg_v', 'sim_forward', 'sim_reverse']

In [ ]:
def extract_subgraph(adj, max_nodes=50000, seed=42):
    """Extract connected subgraph via BFS from high-degree nodes."""
    rng = np.random.default_rng(seed)
    out_deg = np.array(adj.sum(axis=1)).flatten()
    in_deg = np.array(adj.sum(axis=0)).flatten()
    total_deg = out_deg + in_deg
    
    start_node = rng.choice(np.argsort(total_deg)[-100:])
    visited = set([start_node])
    queue = [start_node]
    adj_sym = adj + adj.T
    
    while len(visited) < max_nodes and queue:
        node = queue.pop(0)
        for neighbor in adj_sym[node].indices:
            if neighbor not in visited and len(visited) < max_nodes:
                visited.add(neighbor)
                queue.append(neighbor)
    
    node_indices = np.array(sorted(visited))
    return adj[node_indices][:, node_indices], node_indices

def conformal_quantile(scores, alpha):
    n = len(scores)
    k = int(np.ceil((n + 1) * (1 - alpha)))
    return np.inf if k > n else np.sort(scores)[k - 1]

def get_node_features(adj, n_components=32):
    out_deg = np.array(adj.sum(axis=1)).flatten()
    in_deg = np.array(adj.sum(axis=0)).flatten()
    svd = SVD(n_components=n_components)
    src_emb = svd.fit_transform(adj)
    tgt_emb = svd.embedding_col_
    return out_deg, in_deg, src_emb, tgt_emb

def compute_pair_features(pairs, out_deg, in_deg, src_emb, tgt_emb):
    u, v = pairs[:, 0], pairs[:, 1]
    
    vec_u_src, vec_v_tgt = src_emb[u], tgt_emb[v]
    norm_u = np.linalg.norm(vec_u_src, axis=1)
    norm_v = np.linalg.norm(vec_v_tgt, axis=1)
    norm_u[norm_u == 0], norm_v[norm_v == 0] = 1e-9, 1e-9
    sim_fwd = np.sum(vec_u_src * vec_v_tgt, axis=1) / (norm_u * norm_v)
    
    vec_v_src, vec_u_tgt = src_emb[v], tgt_emb[u]
    norm_v2 = np.linalg.norm(vec_v_src, axis=1)
    norm_u2 = np.linalg.norm(vec_u_tgt, axis=1)
    norm_v2[norm_v2 == 0], norm_u2[norm_u2 == 0] = 1e-9, 1e-9
    sim_rev = np.sum(vec_v_src * vec_u_tgt, axis=1) / (norm_v2 * norm_u2)
    
    return np.column_stack([out_deg[u], in_deg[v], in_deg[u], out_deg[v], sim_fwd, sim_rev])

In [ ]:
def run_pipeline(adj, name, alpha=0.10, n_runs=5, n_components=32, msg_ratio=0.7, node_names=None):
    print(f"\n{'='*70}")
    print(f"Pipeline: {name.upper()} | {n_runs} runs | alpha={alpha}")
    print(f"{'='*70}")
    
    n_nodes = adj.shape[0]
    rows, cols = adj.nonzero()
    all_edges = np.column_stack([rows, cols])
    all_edges_set = set(zip(rows, cols))
    print(f"Graph: {n_nodes:,} nodes, {len(all_edges):,} edges")
    
    run_results, last_run = [], {}
    
    for seed in range(n_runs):
        rng = np.random.default_rng(seed)
        edges = all_edges.copy()
        rng.shuffle(edges)
        
        n = len(edges)
        n_cal, n_test = int(n * 0.1), int(n * 0.1)
        E_cal, E_test, E_train = edges[:n_cal], edges[n_cal:n_cal+n_test], edges[n_cal+n_test:]
        
        n_msg = int(len(E_train) * msg_ratio)
        idx = rng.permutation(len(E_train))
        E_msg, E_sup = E_train[idx[:n_msg]], E_train[idx[n_msg:]]
        
        adj_msg = sp.csr_matrix((np.ones(len(E_msg)), (E_msg[:,0], E_msg[:,1])), shape=(n_nodes, n_nodes))
        adj_msg.setdiag(0); adj_msg.eliminate_zeros()
        
        out_deg, in_deg, src_emb, tgt_emb = get_node_features(adj_msg, n_components)
        
        def make_dataset(pos_edges):
            neg = set()
            while len(neg) < len(pos_edges):
                u, v = rng.integers(0, n_nodes, 2)
                if u != v and (u, v) not in all_edges_set: neg.add((u, v))
            neg = np.array(list(neg))
            X = np.vstack([compute_pair_features(pos_edges, out_deg, in_deg, src_emb, tgt_emb),
                          compute_pair_features(neg, out_deg, in_deg, src_emb, tgt_emb)])
            y = np.hstack([np.ones(len(pos_edges)), np.zeros(len(neg))])
            return X, y, np.vstack([pos_edges, neg])
        
        X_train, y_train, _ = make_dataset(E_sup)
        X_cal, y_cal, _ = make_dataset(E_cal)
        X_test, y_test, test_pairs = make_dataset(E_test)
        
        clf = GradientBoostingClassifier(random_state=seed, n_estimators=100)
        clf.fit(X_train, y_train)
        
        p_cal, p_test = clf.predict_proba(X_cal)[:,1], clf.predict_proba(X_test)[:,1]
        t_lower = conformal_quantile(p_cal[y_cal == 0], alpha)
        t_upper = 1 - conformal_quantile(1 - p_cal[y_cal == 1], alpha)
        
        sets = []
        for p in p_test:
            s = set()
            if p <= t_lower: s.add(0)
            if p >= t_upper: s.add(1)
            sets.append(s if s else {0, 1})
        
        coverage = np.mean([y in s for y, s in zip(y_test, sets)])
        abstention = np.mean([len(s) > 1 for s in sets])
        
        decided_idx = [i for i, s in enumerate(sets) if len(s) == 1]
        f1_dec = 0.0
        if decided_idx:
            _, _, f1_dec, _ = precision_recall_fscore_support(
                y_test[decided_idx], [list(sets[i])[0] for i in decided_idx], average='binary', zero_division=0)
        
        auc = roc_auc_score(y_test, p_test)
        y_pred = (p_test >= 0.5).astype(int)
        _, _, base_f1, _ = precision_recall_fscore_support(y_test, y_pred, average='binary', zero_division=0)
        
        run_results.append({"AUC": auc, "Base F1": base_f1, "Coverage": coverage, 
                           "Abstention": abstention, "Decided F1": f1_dec})
        last_run = {'clf': clf, 'y_test': y_test, 'p_test': p_test, 'y_pred': y_pred,
                   'test_pairs': test_pairs, 't_lower': t_lower, 't_upper': t_upper, 
                   'sets': sets, 'auc': auc, 'node_names': node_names}
        
        print(f"Run {seed+1}: AUC={auc:.4f} | Cov={coverage:.4f} | Abs={abstention:.4f}")
    
    df = pd.DataFrame(run_results)
    print(f"\n--- Results ({name}) ---")
    for col in df.columns:
        print(f"{col}: {df[col].mean():.4f} +/- {df[col].std():.4f}")
    
    # Feature Importance
    fi = last_run['clf'].feature_importances_
    print(f"\n--- Feature Importance ---")
    for i in np.argsort(fi)[::-1]:
        print(f"{FEATURE_NAMES[i]}: {fi[i]:.4f}")
    
    # Error Analysis
    y_test, p_test, y_pred = last_run['y_test'], last_run['p_test'], last_run['y_pred']
    test_pairs, names = last_run['test_pairs'], last_run['node_names']
    
    fp_idx = np.where((y_test == 0) & (y_pred == 1))[0]
    fn_idx = np.where((y_test == 1) & (y_pred == 0))[0]
    fp_sorted = fp_idx[np.argsort(p_test[fp_idx])[::-1]] if len(fp_idx) > 0 else []
    fn_sorted = fn_idx[np.argsort(p_test[fn_idx])] if len(fn_idx) > 0 else []
    
    print(f"\n--- Error Analysis ---")
    print(f"False Positives: {len(fp_idx):,} | False Negatives: {len(fn_idx):,}")
    
    print(f"\nTop 10 False Positives:")
    for rank, idx in enumerate(fp_sorted[:10], 1):
        pair = test_pairs[idx]
        nm = f"{names[pair[0]]} -> {names[pair[1]]}" if names is not None else f"{pair[0]}->{pair[1]}"
        print(f"{rank}. {nm} (p={p_test[idx]:.4f})")
    
    print(f"\nTop 10 False Negatives:")
    for rank, idx in enumerate(fn_sorted[:10], 1):
        pair = test_pairs[idx]
        nm = f"{names[pair[0]]} -> {names[pair[1]]}" if names is not None else f"{pair[0]}->{pair[1]}"
        print(f"{rank}. {nm} (p={p_test[idx]:.4f})")
    
    # Plots
    fig, axes = plt.subplots(2, 3, figsize=(15, 9))
    fig.suptitle(f'{name.upper()} - Results', fontsize=14, fontweight='bold')
    
    fpr, tpr, _ = roc_curve(y_test, p_test)
    axes[0,0].plot(fpr, tpr, 'b-', lw=2, label=f'AUC={last_run["auc"]:.3f}')
    axes[0,0].plot([0,1], [0,1], 'k--')
    axes[0,0].set_xlabel('FPR'); axes[0,0].set_ylabel('TPR')
    axes[0,0].set_title('ROC Curve'); axes[0,0].legend()
    
    axes[0,1].hist(p_test[y_test==0], bins=30, alpha=0.5, color='red', label='Non-edge')
    axes[0,1].hist(p_test[y_test==1], bins=30, alpha=0.5, color='blue', label='Edge')
    axes[0,1].axvline(last_run['t_lower'], color='red', ls='--')
    axes[0,1].axvline(last_run['t_upper'], color='blue', ls='--')
    axes[0,1].set_title('Probability Distribution'); axes[0,1].legend()
    
    sizes = [len(s) for s in last_run['sets']]
    axes[0,2].bar(['Decided', 'Abstain'], [sizes.count(1), sizes.count(2)], color=['green', 'orange'])
    axes[0,2].set_title('Prediction Sets')
    
    sorted_idx = np.argsort(fi)[::-1]
    axes[1,0].barh([FEATURE_NAMES[i] for i in sorted_idx[::-1]], [fi[i] for i in sorted_idx[::-1]])
    axes[1,0].set_title('Feature Importance')
    
    if len(fp_idx) > 0: axes[1,1].hist(p_test[fp_idx], bins=20, alpha=0.5, color='orange', label=f'FP')
    if len(fn_idx) > 0: axes[1,1].hist(p_test[fn_idx], bins=20, alpha=0.5, color='purple', label=f'FN')
    axes[1,1].axvline(0.5, color='k', ls='--')
    axes[1,1].set_title('Error Distribution'); axes[1,1].legend()
    
    axes[1,2].bar(['FP', 'FN'], [len(fp_idx), len(fn_idx)], color=['orange', 'purple'])
    axes[1,2].set_title('Error Counts')
    
    plt.tight_layout()
    plt.show()
    
    return {'name': name, 'df': df, 'means': df.mean().to_dict(), 
            'feature_importance': dict(zip(FEATURE_NAMES, fi.tolist())),
            'n_fp': len(fp_idx), 'n_fn': len(fn_idx)}

In [ ]:
def load_and_prepare(dataset_name, max_nodes=None):
    """Load dataset and optionally extract subgraph."""
    print(f"\nLoading {dataset_name}...")
    dataset = load_netset(dataset_name)
    adj = dataset['adjacency']
    print(f"Full graph: {adj.shape[0]:,} nodes, {adj.nnz:,} edges")
    
    node_names = dataset.get('names', None)
    
    if max_nodes and adj.shape[0] > max_nodes:
        print(f"Extracting subgraph ({max_nodes:,} nodes)...")
        adj, node_idx = extract_subgraph(adj, max_nodes=max_nodes, seed=RNG_SEED)
        if node_names is not None:
            node_names = node_names[node_idx]
    
    adj.setdiag(0)
    adj.eliminate_zeros()
    adj = adj.tocsr()
    
    print(f"Using: {adj.shape[0]:,} nodes, {adj.nnz:,} edges")
    return adj, node_names

In [ ]:
# Configuration
DATASETS = [
    ('wikivitals', None),      # ~10K nodes - use full
    ('wikischools', None),     # ~4K nodes - use full  
    ('wikihumans', 40000),     # ~1M nodes - subsample
    ('wikilinks', 30000),      # ~3.2M nodes - subsample
]

N_RUNS = 5
ALPHA = 0.10
N_COMPONENTS = 32

all_results = []

In [ ]:
# Run WikiVitals
adj, names = load_and_prepare('wikivitals')
result = run_pipeline(adj, 'WikiVitals', alpha=ALPHA, n_runs=N_RUNS, n_components=N_COMPONENTS, node_names=names)
all_results.append(result)

In [ ]:
# Run WikiSchools
adj, names = load_and_prepare('wikischools')
result = run_pipeline(adj, 'WikiSchools', alpha=ALPHA, n_runs=N_RUNS, n_components=N_COMPONENTS, node_names=names)
all_results.append(result)

In [ ]:
# Run WikiHumans (subsampled)
adj, names = load_and_prepare('wikihumans', max_nodes=40000)
result = run_pipeline(adj, 'WikiHumans', alpha=ALPHA, n_runs=N_RUNS, n_components=N_COMPONENTS, node_names=names)
all_results.append(result)

In [ ]:
# Run WikiLinks (subsampled)
adj, names = load_and_prepare('wikilinks', max_nodes=30000)
result = run_pipeline(adj, 'WikiLinks', alpha=ALPHA, n_runs=N_RUNS, n_components=N_COMPONENTS, node_names=names)
all_results.append(result)

In [ ]:
# Comparison Summary
print("\n" + "="*80)
print("COMPARISON ACROSS ALL DATASETS")
print("="*80)

comparison_data = []
for r in all_results:
    row = {'Dataset': r['name']}
    row.update(r['means'])
    comparison_data.append(row)

comparison_df = pd.DataFrame(comparison_data)
print(comparison_df.to_string(index=False))

In [ ]:
# Comparison Plots
fig, axes = plt.subplots(1, 4, figsize=(16, 4))

datasets = [r['name'] for r in all_results]
metrics = ['AUC', 'Coverage', 'Abstention', 'Decided F1']
colors = ['steelblue', 'green', 'orange', 'purple']

for ax, metric, color in zip(axes, metrics, colors):
    values = [r['means'][metric] for r in all_results]
    bars = ax.bar(datasets, values, color=color, alpha=0.7)
    ax.set_title(metric)
    ax.set_ylim(0, 1.05)
    for bar, val in zip(bars, values):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.02, f'{val:.3f}', ha='center', fontsize=9)
    ax.tick_params(axis='x', rotation=45)

plt.suptitle('Comparison Across All Wiki Datasets', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# Feature Importance Comparison
fig, ax = plt.subplots(figsize=(12, 6))

x = np.arange(len(FEATURE_NAMES))
width = 0.2
colors = ['#1f77b4', '#ff7f0e', '#2ca02c', '#d62728']

for i, r in enumerate(all_results):
    fi = [r['feature_importance'][f] for f in FEATURE_NAMES]
    ax.bar(x + i*width, fi, width, label=r['name'], color=colors[i], alpha=0.8)

ax.set_xlabel('Features')
ax.set_ylabel('Importance')
ax.set_title('Feature Importance Comparison Across Datasets')
ax.set_xticks(x + width * 1.5)
ax.set_xticklabels(FEATURE_NAMES, rotation=45, ha='right')
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
# Final Summary Table
print("\nFinal Summary:")
display(comparison_df.round(4))